In [6]:
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn
import pandas as pd
import numpy as np
from hasoc_model import encode_labels
%load_ext autoreload
%autoreload 2

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
df_clara = pd.read_csv("../hasoc_model/hasoc_dataset/train.tsv", sep="\t")
df_clara.columns = ["id", "text", "label_A", "label_B", "label_C"]
df_clara = df_clara[["text", "label_A", "label_B", "label_C"]] 
df_clara = encode_labels(df_clara)
#df_clara = df_clara[0:100]
print(df_clara.head())

                                                text label_A label_B label_C  \
0  #DhoniKeepsTheGlove | WATCH: Sports Minister K...     NOT    NONE    NONE   
1  @politico No. We should remember very clearly ...     HOF    HATE     TIN   
2  @cricketworldcup Guess who would be the winner...     NOT    NONE    NONE   
3  Corbyn is too politically intellectual for #Bo...     NOT    NONE    NONE   
4  All the best to #TeamIndia for another swimmin...     NOT    NONE    NONE   

   label_A_enc  label_B_enc  label_C_enc  
0            0          NaN          NaN  
1            1          0.0          1.0  
2            0          NaN          NaN  
3            0          NaN          NaN  
4            0          NaN          NaN  


In [8]:
class Paola(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_outputs=8, bin_outputs=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_outputs)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, bin_outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.regressor(pooled), self.classifier(pooled)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_paola = Paola().to(device)
model_paola.load_state_dict(torch.load("../paola/model2_loaded.pth", map_location=device, weights_only=True))

print("model2_loaded.pth loaded and ready to use!")

tokenizer_paola = AutoTokenizer.from_pretrained("distilbert-base-uncased")

model2_loaded.pth loaded and ready to use!


In [10]:
encodings_paola = tokenizer_paola(df_clara["text"].tolist(), truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paola = encodings_paola['input_ids'].to(device)
attention_mask_paola = encodings_paola['attention_mask'].to(device)

with torch.no_grad():
    preds_num, preds_bin = model_paola(input_ids=input_ids_paola, attention_mask=attention_mask_paola)

preds_num = preds_num.cpu().numpy()
preds_bin = preds_bin.cpu().numpy()
preds_bin = (preds_bin > 0.5).astype(int)

for idx in range (3):
    print(f"Sentence: {df_clara["text"].tolist()[idx]}")
    print(f"Numerical predictions: {preds_num[idx]}")
    print(f"Binary predictions: {preds_bin[idx]}")
    print()


Sentence: #DhoniKeepsTheGlove | WATCH: Sports Minister Kiren Rijiju issues statement backing MS Dhoni over 'Balidaan Badge', tells BCCI to take up the matter with ICC and keep government in the know as nation's pride is involved    https://t.co/zuo5335Rjr
Numerical predictions: [1.4261338  1.026587   0.7156262  0.6578899  1.8180861  0.5279458
 1.3764595  0.03854389]
Binary predictions: [0 0 0 0 1]

Sentence: @politico No. We should remember very clearly that #Individual1 just admitted to treason . #TrumpIsATraitor  #McCainsAHero #JohnMcCainDay
Numerical predictions: [2.8258026  2.305354   1.8951534  1.739517   2.1474311  1.330773
 2.071613   0.32836258]
Binary predictions: [0 0 0 0 0]

Sentence: @cricketworldcup Guess who would be the winner of this #CWC19?     Team who gets maximum points from the abandoned matches 😄 #ShameOnICC #WIvsENG @ICC
Numerical predictions: [2.2002656  1.9434769  1.5539637  1.3744226  2.0864556  0.9715349
 1.9307483  0.07670165]
Binary predictions: [1 0 0 0 0]

In [11]:
new_feature_names = ['sentiment', 'respect', 'insult', 'humiliate', 'status',
                  'dehumanize', 'attack_defend', 'hatespeech',
                     'target_race', 'target_religion', 'target_origin', 'target_gender',
                'target_sexuality']

combined_preds = np.concatenate([preds_num, preds_bin], axis=1)
preds_df = pd.DataFrame(combined_preds, columns=new_feature_names)
df_clara = pd.concat([df_clara.reset_index(drop=True), preds_df], axis=1)

df_clara.to_csv("../hasoc_model/hasoc_dataset/hasoc_dataset_with_features_train.tsv", sep='\t', index=False)